# Notebook 2 Supplement: Measurement-Based Quantum Computing (MBQC)

This notebook is part of the FDP hands-on session on MBQC (One-Way Quantum Computing). It covers:
1. **Graph and Cluster States**: Preparing 1D chain and 2D grid cluster states.
2. **State Teleportation & Single-Qubit Identity**: Showing how single-qubit measurement propagates state information.
3. **Single-Qubit Rotations with Feed-Forward**: Implementing adaptive measurement angles depending on previous measurement outcomes.

We use **Qiskit** and **PennyLane** to simulate and verify these MBQC processes.

## Section 1: Creating Graph and Cluster States

MBQC starts with an entangled state represented by a graph $G = (V,E)$, where vertices $V$ are qubits initialized in $|+\rangle$ and edges $E$ correspond to Controlled-Z (CZ) gates applied between them.

In [1]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

# 1. Create a 4-qubit 1D Cluster State Chain
qc_1d = QuantumCircuit(4)
qc_1d.h([0, 1, 2, 3])
qc_1d.cz(0, 1)
qc_1d.cz(1, 2)
qc_1d.cz(2, 3)

print("1D Cluster Chain Circuit:")
print(qc_1d.draw(output='text'))

# 2. Create a 2x2 2D Grid Cluster State
qc_2d = QuantumCircuit(4)
qc_2d.h([0, 1, 2, 3])
# Grid edges: (0,1), (2,3) horizontally; (0,2), (1,3) vertically
qc_2d.cz(0, 1)
qc_2d.cz(2, 3)
qc_2d.cz(0, 2)
qc_2d.cz(1, 3)

print("\n2D Grid Cluster Circuit:")
print(qc_2d.draw(output='text'))

1D Cluster Chain Circuit:
     ┌───┐         
q_0: ┤ H ├─■───────
     ├───┤ │       
q_1: ┤ H ├─■──■────
     ├───┤    │    
q_2: ┤ H ├────■──■─
     ├───┤       │ 
q_3: ┤ H ├───────■─
     └───┘         

2D Grid Cluster Circuit:
     ┌───┐         
q_0: ┤ H ├─■──■────
     ├───┤ │  │    
q_1: ┤ H ├─■──┼──■─
     ├───┤    │  │ 
q_2: ┤ H ├─■──■──┼─
     ├───┤ │     │ 
q_3: ┤ H ├─■─────■─
     └───┘         


### Cluster States in PennyLane

In [2]:
import pennylane as qml

dev_cluster = qml.device("default.qubit", wires=4)

@qml.qnode(dev_cluster)
def prepare_1d_cluster():
    # Initialize all in |+>
    for i in range(4):
        qml.Hadamard(wires=i)
    # Entangle along chain
    qml.CZ(wires=[0, 1])
    qml.CZ(wires=[1, 2])
    qml.CZ(wires=[2, 3])
    return qml.state()

state_1d = prepare_1d_cluster()
print("PennyLane 1D Cluster Statevector (first 4 components):")
print(np.round(state_1d[:4], 4))

PennyLane 1D Cluster Statevector (first 4 components):
[ 0.25+0.j  0.25+0.j  0.25+0.j -0.25+0.j]


## Section 2: State Teleportation & Single-Qubit Identity

The basic building block of MBQC is state propagation. By preparing a 2-qubit cluster state, placing the input state on qubit 0, and measuring qubit 0 in the $X$-basis, the state is "teleportation-propagated" to qubit 1.

Depending on the measurement outcome $s_0 \in \{0, 1\}$ of qubit 0, the state of qubit 1 is:
$$|\psi_{\text{out}}\rangle = X^{s_0} H |\psi_{\text{in}}\rangle$$
where $H$ is the Hadamard gate, and $X^{s_0}$ is the Pauli by-product (correction) operator.

In [3]:
# We prepare an arbitrary input state |psi_in> = RY(pi/3)|0> on Q0
theta = np.pi / 3
qc_tele = QuantumCircuit(2)
qc_tele.ry(theta, 0)

# Prepare cluster entanglement (CZ after putting target in |+>)
qc_tele.h(1)
qc_tele.cz(0, 1)

# Measure Q0 in X-basis (apply H then measure)
qc_tele.h(0)

# Inspect the full statevector to analyze output states
sv_tele = Statevector.from_instruction(qc_tele).data

# In Qiskit (little-endian: q1 q0):
# Outcome 0 on Q0 corresponds to indices 0 and 2
out_0 = np.array([sv_tele[0], sv_tele[2]])
out_0 = out_0 / np.linalg.norm(out_0)

# Outcome 1 on Q0 corresponds to indices 1 and 3
out_1 = np.array([sv_tele[1], sv_tele[3]])
out_1 = out_1 / np.linalg.norm(out_1)

# Expected outputs:
# For outcome 0: H|psi_in> = H RY(pi/3)|0>
# For outcome 1: X H|psi_in> = X H RY(pi/3)|0>
qc_ideal = QuantumCircuit(1)
qc_ideal.ry(theta, 0)
qc_ideal.h(0)
ideal_state = Statevector.from_instruction(qc_ideal).data

print("Ideal Target State H|psi_in>:", ideal_state)
print("\nMBQC Q1 State (Q0 measured 0):", out_0)
print("Fidelity for outcome 0:", np.abs(np.vdot(ideal_state, out_0))**2)

print("\nMBQC Q1 State (Q0 measured 1):", out_1)
# We apply Pauli X to correct the by-product operator for outcome 1
corrected_out_1 = np.array([out_1[1], out_1[0]])
print("Corrected Q1 State (Pauli-X applied):", corrected_out_1)
print("Fidelity for outcome 1:", np.abs(np.vdot(ideal_state, corrected_out_1))**2)

Ideal Target State H|psi_in>: [0.96592583+0.j 0.25881905+0.j]

MBQC Q1 State (Q0 measured 0): [0.96592583+0.j 0.25881905+0.j]
Fidelity for outcome 0: 0.9999999999999998

MBQC Q1 State (Q0 measured 1): [0.25881905+0.j 0.96592583+0.j]
Corrected Q1 State (Pauli-X applied): [0.96592583+0.j 0.25881905+0.j]
Fidelity for outcome 1: 0.9999999999999998


## Section 3: Single-Qubit Rotations with Feed-Forward

To perform computation, we measure qubits at specific angles in the $XY$-plane:
$$|\theta_\pm\rangle = \frac{|0\rangle \pm e^{i\theta}|1\rangle}{\sqrt{2}}$$
This implements a rotation $R_z(-\theta)$ followed by a Hadamard $H$ on the input qubit state:
$$|\psi_{\text{out}}\rangle = X^s H R_z(-\theta) |\psi_{\text{in}}\rangle$$

If we have multiple rotations in a chain (e.g. $R_z(-\theta)$ then $R_x(-\phi)$), the outcome $s_0$ of the first measurement introduces a Pauli $X^{s_0}$ by-product on the next stage. Because $X$ maps a rotation angle $\phi$ to $-\phi$, we must adjust the second measurement angle dynamically (feed-forward):
$$\phi_{\text{measured}} = (-1)^{s_0} \phi$$

Let's simulate this 3-qubit rotation chain with feed-forward in Qiskit.

In [4]:
from qiskit_aer import AerSimulator
from qiskit import QuantumRegister, ClassicalRegister

# Parameters for the desired rotations
theta = np.pi / 4  # First rotation angle (Z-rotation)
phi = np.pi / 3    # Second rotation angle (X-rotation)

# Setup 3-qubit register and classical registers for feed-forward
qr = QuantumRegister(3)
cr0 = ClassicalRegister(1, name="cr0")
cr1 = ClassicalRegister(1, name="cr1")
cr2 = ClassicalRegister(1, name="cr2")
qc_rot = QuantumCircuit(qr, cr0, cr1, cr2)

# Step 1: Prepare the 3-qubit cluster chain
qc_rot.h(qr)
qc_rot.cz(0, 1)
qc_rot.cz(1, 2)

# Step 2: Measure Q0 at angle theta in XY plane
# (apply Rz(-theta) then H then measure)
qc_rot.rz(-theta, 0)
qc_rot.h(0)
qc_rot.measure(0, cr0)

# Step 3: Measure Q1 at angle phi, conditioned on Q0's measurement outcome
# If cr0 = 0: measure at -phi (Rz(phi) then H)
# If cr0 = 1: measure at phi (Rz(-phi) then H)
# Under the hood, we can implement this with controlled rotations:
# Rz( (-1)^s0 * -phi ) = Rz( -phi * (1 - 2*s0) )
# We implement this dynamic angle change using a class of classically controlled gates in Qiskit:
with qc_rot.if_test((cr0, 0)):
    qc_rot.rz(phi, 1)  # Measure at -phi (Rz(phi) before H)
with qc_rot.if_test((cr0, 1)):
    qc_rot.rz(-phi, 1) # Measure at +phi (Rz(-phi) before H)

qc_rot.h(1)
qc_rot.measure(1, cr1)

# Measure Q2 at the end in Z basis
qc_rot.measure(2, cr2)

print("MBQC Rotation Chain with Feed-Forward:")
print(qc_rot.draw(output='text'))

# Simulate counts
sim_rot = AerSimulator()
counts_rot = sim_rot.run(qc_rot, shots=1000).result().get_counts()
print("\nSampled measurement outcomes (cr2 cr1 cr0):")
print(counts_rot)

MBQC Rotation Chain with Feed-Forward:
       ┌───┐   ┌──────────┐┌───┐┌─┐                                     »
 q0_0: ┤ H ├─■─┤ Rz(-π/4) ├┤ H ├┤M├─────────────────────────────────────»
       ├───┤ │ └──────────┘└───┘└╥┘┌────── ┌─────────┐ ───────┐ ┌────── »
 q0_1: ┤ H ├─■──────■────────────╫─┤ If-0  ┤ Rz(π/3) ├  End-0 ├─┤ If-0  »
       ├───┤        │       ┌─┐  ║ └──╥─── └─────────┘ ───────┘ └──╥─── »
 q0_2: ┤ H ├────────■───────┤M├──╫────╫────────────────────────────╫────»
       └───┘                └╥┘  ║ ┌──╨──┐                      ┌──╨──┐ »
cr0: 1/══════════════════════╬═══╩═╡ 0x0 ╞══════════════════════╡ 0x1 ╞═»
                             ║   0 └─────┘                      └─────┘ »
cr1: 1/══════════════════════╬══════════════════════════════════════════»
                             ║                                          »
cr2: 1/══════════════════════╩══════════════════════════════════════════»
                             0                                          »